In [1]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

In [2]:
import numpy as np
import pandas as pd
import SimpleITK as sitk

from lib import *

In [3]:
manifest_path = "/home/furkan/projects/mebar-radiology/data/manifest.csv"
manifest_df = pd.read_csv(manifest_path)

# Health Checks
We have a dataset of 3d MR imagings and masks for two modalities. Before applying any processing to these, we need to check if there exists any meaningless values - except nulls and duplicates, we are focusing on more in context anomalies - such as a segmentation mask has to be a binary array, so here we need to check if there exists any mask with non-binary values - then fix.

## 1. Segmentation Masks Non-Binary Check
A segmentation mask must consist of only zeroes and ones. Now we will check if there exists any mask with non-binary values.

In [4]:
invalid_count = 0
for idx, row in manifest_df.iterrows():
    mask_path = get_abs_path(row["mask_path"])
    mask_image = sitk.ReadImage(mask_path)
    mask_array = sitk.GetArrayFromImage(mask_image)
    # filter of non-binary values
    invalids_np_mask = (mask_array != 0) & (mask_array != 1)
    # apply filter, returns a 1d np array of non-binary values
    invalids = mask_array[invalids_np_mask]
    if invalids.size > 0:
        print(f"sample {row['sample_id']} has invalid mask values: ", invalids)
        invalid_count += 1

print(f"Health check complete with {invalid_count} invalid samples detected.")

Health check complete with 0 invalid samples detected.


## 2. RAS+ Coordinates Standard Check

All imagings should be standardized to one coordinate system. RAS+ means center of machine is 0,0,0 and right, up and front goes positive. Across my work, I want all the imagings to be aligned in RAS+ space.

In [6]:
image_spaces = manifest_df["image_space"]
image_spaces.unique()

<StringArray>
['left-posterior-superior']
Length: 1, dtype: str

In [7]:
mask_spaces = manifest_df["mask_space"]
mask_spaces.unique()

<StringArray>
['left-posterior-superior']
Length: 1, dtype: str

All samples are aligned to LPS space. Then a realigning to RAS+ will be necessary.